# Anomaly Detection

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json

In [ ]:
fm_encoded = pd.read_csv('../features/fm_encoded.csv', index_col='user_id')
transactions_df = pd.read_csv('../features/transactions_enriched.csv', parse_dates=['date'])

## Feature Selection

- Velocity Features: how fast the user spending

Transactions in last 1 day, 7 days, and 30 days, total count last 30 days, and all-time transaction count
- Amount Features: how large or unusual thier spend values are

Average transaction amount last 30 days, all-time mean transaction amount, largest single transaction ever, average monthly spend, highest single-month spend, lifetime total spend, current balance
- Recency Features: how long since last activity

Days since last transaction, recency in seconds
- Merchant: new merchants

Merchant Diversity, count of unique merchants visited, count of first-time merchants

In [ ]:
VELOCITY_FEATURES = [
    'vel_1d',                            
    'vel_7d',                        
    'vel_30d',                           
    'count_30d',                         
    'accounts.COUNT(transactions)',      
]

AMOUNT_FEATURES = [
    'avg_amt_30d',                                  
    'accounts.MEAN(transactions.amount)',            
    'accounts.MAX(transactions.amount)',             
    'accounts.MEAN(monthly_stats.total_spend)',      
    'accounts.MAX(monthly_stats.total_spend)',      
    'accounts.SUM(monthly_stats.total_spend)',       
    'accounts.balances_current',                     
]

RECENCY_FEATURES = [
    'days_since_last',                                       
    'accounts.TIME_SINCE_LAST(transactions.date)',           
]

MERCHANT_FEATURES = [
    'merchant_diversity',    
    'new_merchant_count',   
]

ALL_ANOMALY_FEATURES = VELOCITY_FEATURES + AMOUNT_FEATURES + RECENCY_FEATURES + MERCHANT_FEATURES

In [ ]:
for name, group in [('Velocity', VELOCITY_FEATURES), ('Amount', AMOUNT_FEATURES),
                     ('Recency', RECENCY_FEATURES), ('Merchant', MERCHANT_FEATURES)]:
    found = [c for c in group if c in fm_encoded.columns]
    print(f'  {name} ({len(found)}): {found}')

### Missing Values

In [ ]:
available_features = [f for f in ALL_ANOMALY_FEATURES if f in fm_encoded.columns]
missing_features   = [f for f in ALL_ANOMALY_FEATURES if f not in fm_encoded.columns]

X = fm_encoded[available_features].fillna(0).copy()
X.index = fm_encoded.index

## Scalling and Train Model

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Isolation Forest

In [ ]:
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,   
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)
iso_labels = iso_forest.fit_predict(X_scaled)  
iso_scores = iso_forest.score_samples(X_scaled) 

print(f'Isolation Forest — flagged {(iso_labels == -1).sum()} users')

## Local Outlier Factor

In [ ]:
lof = LocalOutlierFactor(
    n_neighbors=min(20, len(X) - 1),
    contamination=0.05,
    metric='euclidean',
    n_jobs=-1
)
lof_labels = lof.fit_predict(X_scaled)
lof_scores = lof.negative_outlier_factor_

print(f'LOF — flagged {(lof_labels == -1).sum()} users')

## Ensemble & Score Normalization

In [ ]:
anomaly_df = pd.DataFrame(index=fm_encoded.index)

anomaly_df['iso_flag'] = iso_labels == -1
anomaly_df['lof_flag'] = lof_labels == -1

anomaly_df['is_anomaly'] = anomaly_df['iso_flag'] & anomaly_df['lof_flag']

iso_norm = 1 - (iso_scores - iso_scores.min()) / (iso_scores.max() - iso_scores.min())
lof_norm = 1 - (lof_scores - lof_scores.min()) / (lof_scores.max() - lof_scores.min())

anomaly_df['iso_score'] = (iso_norm * 100).round(2)
anomaly_df['lof_score'] = (lof_norm * 100).round(2)
anomaly_df['anomaly_score'] = ((iso_norm + lof_norm) / 2 * 100).round(2)

def get_severity(row):
    if not row['is_anomaly']: return 'Normal'
    if row['anomaly_score'] >= 80: return 'Critical'
    if row['anomaly_score'] >= 65: return 'High'
    return 'Medium'

anomaly_df['severity'] = anomaly_df.apply(get_severity, axis=1)

for col in available_features:
    anomaly_df[col] = fm_encoded[col]

context_cols = ['task_segment', 'top_category', 'top_merchant', 'primary_spending_trend']
for col in context_cols:
    if col in fm_encoded.columns:
        anomaly_df[col] = fm_encoded[col]

category_cols = [
    c for c in fm_encoded.columns
    if 'monthly_stats' in c
    and any(x in c for x in ['MAX', 'MEAN', 'SUM'])
    and 'total_spend' not in c
    and 'total_transactions' not in c
    and 'avg_transaction' not in c
]
for col in category_cols:
    anomaly_df[col] = fm_encoded[col]

ensemble_count = anomaly_df['is_anomaly'].sum()
print(f'Ensemble complete')
print(f'ISO Forest flagged: {anomaly_df["iso_flag"].sum()}')
print(f'LOF flagged: {anomaly_df["lof_flag"].sum()}')
print(f'Final (both agreed): {ensemble_count} ({ensemble_count/len(anomaly_df)*100:.1f}%)')
print(f'\nSeverity breakdown:')
print(anomaly_df['severity'].value_counts())

## Anomaly Driver Detection

In [ ]:
FEATURE_LABELS = {
    'vel_1d': 'Transactions today',
    'vel_7d': 'Transactions this week',
    'vel_30d': 'Transactions this month',
    'count_30d': 'Transaction count (30d)',
    'accounts.COUNT(transactions)': 'Total transactions ever',
    'avg_amt_30d': 'Avg transaction amount (30d)',
    'accounts.MEAN(transactions.amount)': 'All-time avg transaction amount',
    'accounts.MAX(transactions.amount)': 'Largest transaction ever',
    'accounts.MEAN(monthly_stats.total_spend)': 'Avg monthly spend',
    'accounts.MAX(monthly_stats.total_spend)': 'Highest single-month spend',
    'accounts.SUM(monthly_stats.total_spend)': 'Lifetime total spend',
    'accounts.balances_current': 'Current balance',
    'days_since_last': 'Days since last transaction',
    'accounts.TIME_SINCE_LAST(transactions.date)':  'Seconds since last transaction',
    'merchant_diversity': 'Unique merchants visited',
    'new_merchant_count': 'New merchants explored',
}

population_medians = X.median()

def identify_drivers(user_id, top_n=3):
    if user_id not in X.index:
        return []

    user_row = X.loc[user_id]
    deviations = {}

    for col in available_features:
        if col not in X.columns:
            continue
        median_val = population_medians[col]
        user_val   = user_row[col]
        deviations[col] = abs(user_val - median_val) / (abs(median_val) + 1e-9)

    top = sorted(deviations.items(), key=lambda x: x[1], reverse=True)[:top_n]

    results = []
    for feat, deviation in top:
        user_val = round(float(user_row[feat]), 2)
        median_val = round(float(population_medians[feat]), 2)
        direction = 'above' if user_val > median_val else 'below'
        label = FEATURE_LABELS.get(feat, feat)
        results.append({
            'feature': feat,
            'label': label,
            'user_value': user_val,
            'population_median': median_val,
            'direction': direction,
            'deviation_x': round(deviation, 2),
            'explanation': f"{label}: {user_val} (median is {median_val} — you are {direction} average by {round(deviation,1)}x)"
        })
    return results

anomaly_df['drivers'] = anomaly_df.index.to_series().apply(identify_drivers)

print('Drivers calculated for all users')
print('\nTop anomalous users:')
anomalous_users = anomaly_df[anomaly_df['is_anomaly']].sort_values('anomaly_score', ascending=False)
display(anomalous_users[['anomaly_score', 'severity', 'vel_7d', 'avg_amt_30d', 'merchant_diversity', 'drivers']].head(10))

## Per-Transaction Anomaly Context

In [ ]:
def get_suspicious_transactions(user_id, transactions_df, n=5):
    user_trans = transactions_df[transactions_df['user_id'] == user_id].copy()
    if user_trans.empty:
        return []

    priority_map = {'Unusually Large': 0, 'Unusually Small': 1, 'Normal': 2}
    user_trans['_priority'] = user_trans['spend_anomaly_type'].map(priority_map).fillna(2)

    result = (
        user_trans
        .sort_values(['_priority', 'deviation_ratio'], ascending=[True, False])
        .head(n)
    )
    return result[['date', 'amount', 'merchant_name', 'category_id', 'deviation_ratio', 'spend_anomaly_type']].to_dict('records')


def get_anomaly_transaction_summary(user_id, transactions_df):
    user_trans = transactions_df[transactions_df['user_id'] == user_id]
    if user_trans.empty:
        return {}
    return user_trans['spend_anomaly_type'].value_counts().to_dict()


print('Transaction context functions defined')
print(f'\nspend_anomaly_type distribution (all users):')
print(transactions_df['spend_anomaly_type'].value_counts())

### Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Anomaly Detection Results', fontsize=16, fontweight='bold', y=1.02)

ax1 = axes[0]
ax1.hist(anomaly_df[~anomaly_df['is_anomaly']]['anomaly_score'],
         bins=30, color='#3498db', alpha=0.7, label='Normal')
ax1.hist(anomaly_df[anomaly_df['is_anomaly']]['anomaly_score'],
         bins=10, color='#e74c3c', alpha=0.9, label='Anomaly')
ax1.set_xlabel('Anomaly Score (0=Normal, 100=Critical)')
ax1.set_ylabel('Users')
ax1.set_title('Score Distribution')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[1]
c_map = anomaly_df['is_anomaly'].map({True: '#e74c3c', False: '#3498db'})
ax2.scatter(anomaly_df['iso_score'], anomaly_df['lof_score'],
            c=c_map, alpha=0.6, s=40, edgecolors='white', linewidth=0.5)
ax2.set_xlabel('Isolation Forest Score')
ax2.set_ylabel('LOF Score')
ax2.set_title('ISO Forest vs LOF\n(Red = flagged by both)')
ax2.legend(handles=[
    mpatches.Patch(color='#3498db', label='Normal'),
    mpatches.Patch(color='#e74c3c', label='Ensemble Anomaly')
])
ax2.grid(alpha=0.3)

ax3 = axes[2]
severity_order  = ['Normal', 'Medium', 'High', 'Critical']
severity_colors = {'Normal': '#3498db', 'Medium': '#f39c12', 'High': '#e67e22', 'Critical': '#e74c3c'}
counts = anomaly_df['severity'].value_counts().reindex(severity_order, fill_value=0)
bars   = ax3.bar(counts.index, counts.values,
                 color=[severity_colors[s] for s in counts.index], edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(val), ha='center', fontweight='bold')
ax3.set_xlabel('Severity')
ax3.set_ylabel('Users')
ax3.set_title('Severity Breakdown')
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Save Outputs

In [ ]:
save_cols = (
    ['iso_score', 'lof_score', 'anomaly_score', 'iso_flag', 'lof_flag', 'is_anomaly', 'severity']
    + available_features
    + [c for c in context_cols if c in anomaly_df.columns]
)
anomaly_df[save_cols].to_csv('outputs/anomaly_scores.csv')
print('Saved to outputs/anomaly_scores.csv')

In [ ]:
def run_anomaly_check(user_id, anomaly_df=anomaly_df, transactions_df=transactions_df):

    if user_id not in anomaly_df.index:
        return {'error': f'User {user_id} not found', 'is_anomaly': False, 'anomaly_score': 0}

    row = anomaly_df.loc[user_id]

    return {
        #Core anomaly verdict
        'user_id': user_id,
        'is_anomaly': bool(row['is_anomaly']),
        'anomaly_score': float(row['anomaly_score']),  
        'severity': str(row['severity']),          
        'iso_flagged': bool(row['iso_flag']),
        'lof_flagged': bool(row['lof_flag']),

        #Velocity
        'transactions_today': int(row.get('vel_1d', 0)),
        'transactions_this_week': int(row.get('vel_7d', 0)),
        'transactions_this_month': int(row.get('vel_30d', 0)),
        'total_transactions_ever': int(row.get('accounts.COUNT(transactions)', 0)),

        #Amount
        'avg_transaction_amt_30d': round(float(row.get('avg_amt_30d', 0)), 2),
        'mean_transaction_all_time': round(float(row.get('accounts.MEAN(transactions.amount)', 0)), 2),
        'largest_transaction_ever': round(float(row.get('accounts.MAX(transactions.amount)', 0)), 2),
        'current_balance': round(float(row.get('accounts.balances_current', 0)), 2),

        #Recency
        'days_since_last_transaction': round(float(row.get('days_since_last', 0)), 1),

        #Merchant behavior
        'unique_merchants_visited': int(row.get('merchant_diversity', 0)),
        'new_merchants_explored': int(row.get('new_merchant_count', 0)),

        #User context
        'spending_segment': str(row.get('task_segment', 'Unknown')),
        'top_category': str(row.get('top_category', 'Unknown')),
        'top_merchant': str(row.get('top_merchant', 'Unknown')),
        'spending_trend': str(row.get('primary_spending_trend', 'Unknown')),

        'anomaly_drivers': [d['explanation'] for d in identify_drivers(user_id, top_n=3)],

        # Top 5 most suspicious transactions ranked by deviation_ratio
        'suspicious_transactions': get_suspicious_transactions(user_id, transactions_df, n=5),

        # Summary of spend_anomaly_type counts across all user transactions
        'transaction_anomaly_summary': get_anomaly_transaction_summary(user_id, transactions_df),
    }

### Tests

In [ ]:
normal_uid = anomaly_df[~anomaly_df['is_anomaly']].index[0]
result_n   = run_anomaly_check(normal_uid)

print(f'NORMAL USER  (ID: {normal_uid})')
print(json.dumps({k: v for k, v in result_n.items() if k != 'suspicious_transactions'}, indent=2))

In [ ]:
anomalous_users = anomaly_df[anomaly_df['is_anomaly']].sort_values('anomaly_score', ascending=False)

if len(anomalous_users) > 0:
    anom_uid = anomalous_users.index[0]
    result_a = run_anomaly_check(anom_uid)

    print(f'ANOMALOUS USER  (ID: {anom_uid})  Severity: {result_a["severity"]}')
    print(json.dumps({k: v for k, v in result_a.items() if k != 'suspicious_transactions'}, indent=2))

    print('\nMost suspicious transactions:')
    if result_a['suspicious_transactions']:
        for t in result_a['suspicious_transactions']:
            print(f"   {t['date']}  {str(t['merchant_name']):25s}  ${t['amount']:>8.2f}  "
                f"{t['spend_anomaly_type']}  (ratio: {round(t['deviation_ratio'], 2)})")
    else:
        print("   No transactions found for this user.")

else:
    print('No anomalous users found. Try increasing contamination to 0.08 in Step 4.')